# use CNN with large dataset to avoid overfitting

In [2]:
import os
import zipfile
import random
import tensorflow as tf
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile

### Getting the data (code is updated in line for windows )

In [3]:
import os
import ssl
import urllib.request
import zipfile

url = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_3367a.zip")

base_dir = os.path.join(os.getcwd(), "data")
os.makedirs(base_dir, exist_ok=True)

local_zip = os.path.join(base_dir, "cats-and-dogs.zip")

# Download (skipped if the file already exists)
if not os.path.exists(local_zip):
    print("Downloading... this is large, so it may take several minutes.")

    # Same idea as wget's --no-check-certificate (only used if needed)
    ctx = ssl.create_default_context()
    # ctx.check_hostname = False
    # ctx.verify_mode = ssl.CERT_NONE

    # Some servers reject Python's default user agent, so pretend to be a browser
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=ctx) as response, open(local_zip, "wb") as out:
        while True:
            chunk = response.read(1024 * 1024)   # 1 MB at a time
            if not chunk:
                break
            out.write(chunk)
    print("Download complete.")

# Extract into the data folder (creates data/PetImages)
if not os.path.exists(os.path.join(base_dir, "PetImages")):
    with zipfile.ZipFile(local_zip, "r") as zip_ref:
        zip_ref.extractall(base_dir)

cat_dir = os.path.join(base_dir, "PetImages", "Cat")
dog_dir = os.path.join(base_dir, "PetImages", "Dog")

print(len(os.listdir(cat_dir)))
print(len(os.listdir(dog_dir)))
# Expected Output:
# 12501
# 12501

12501
12501


### changing file name (new url) 

In [4]:
import os
import urllib.request
import zipfile

url = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")

base_dir = os.path.join(os.getcwd(), "data")
os.makedirs(base_dir, exist_ok=True)

local_zip = os.path.join(base_dir, "cats-and-dogs.zip")

# Download (skipped if the file already exists)
if not os.path.exists(local_zip):
    print("Downloading... about 800 MB, this may take several minutes.")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response, open(local_zip, "wb") as out:
        while True:
            chunk = response.read(1024 * 1024)   # 1 MB at a time
            if not chunk:
                break
            out.write(chunk)
    print("Download complete.")

# Extract (creates data/PetImages)
if not os.path.exists(os.path.join(base_dir, "PetImages")):
    with zipfile.ZipFile(local_zip, "r") as zip_ref:
        zip_ref.extractall(base_dir)

cat_dir = os.path.join(base_dir, "PetImages", "Cat")
dog_dir = os.path.join(base_dir, "PetImages", "Dog")

print(len(os.listdir(cat_dir)))
print(len(os.listdir(dog_dir)))

12501
12501


### preparing the data ( fixed the pathway , split function bug & also added step to skip bad files) 

In [5]:
import os
import random
import shutil
from shutil import copyfile
from PIL import Image

# ---------- Create the folder structure ----------
split_root = os.path.join(base_dir, "cats-v-dogs")

# Start fresh so re-running doesn't mix old and new splits
if os.path.exists(split_root):
    shutil.rmtree(split_root)

TRAINING_CATS_DIR = os.path.join(split_root, "training", "cats")
TRAINING_DOGS_DIR = os.path.join(split_root, "training", "dogs")
TESTING_CATS_DIR  = os.path.join(split_root, "testing", "cats")
TESTING_DOGS_DIR  = os.path.join(split_root, "testing", "dogs")

for d in [TRAINING_CATS_DIR, TRAINING_DOGS_DIR, TESTING_CATS_DIR, TESTING_DOGS_DIR]:
    os.makedirs(d, exist_ok=True)


# ---------- Helper: is this a usable image? ----------
def is_valid_image(path):
    if os.path.getsize(path) == 0:
        return False
    try:
        with Image.open(path) as img:
            img.verify()          # raises an error if the file is corrupt
        return True
    except Exception:
        return False


# ---------- Split function ----------
def split_data(SOURCE, TRAINING, TESTING, SPLIT_SIZE):
    files = []
    for filename in os.listdir(SOURCE):
        file_path = os.path.join(SOURCE, filename)
        if is_valid_image(file_path):
            files.append(filename)
        else:
            print(filename + " is empty or corrupted, so ignoring.")

    training_length = int(len(files) * SPLIT_SIZE)
    shuffled_set = random.sample(files, len(files))
    training_set = shuffled_set[:training_length]
    testing_set = shuffled_set[training_length:]   # fixed: no overlap with training

    for filename in training_set:
        copyfile(os.path.join(SOURCE, filename), os.path.join(TRAINING, filename))

    for filename in testing_set:
        copyfile(os.path.join(SOURCE, filename), os.path.join(TESTING, filename))


# ---------- Run it ----------
split_size = 0.9
split_data(cat_dir, TRAINING_CATS_DIR, TESTING_CATS_DIR, split_size)
split_data(dog_dir, TRAINING_DOGS_DIR, TESTING_DOGS_DIR, split_size)

666.jpg is empty or corrupted, so ignoring.
Thumbs.db is empty or corrupted, so ignoring.
11702.jpg is empty or corrupted, so ignoring.


C:\Users\Asus\anaconda3\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Thumbs.db is empty or corrupted, so ignoring.


In [6]:
# ---------- Check the result ----------
print(len(os.listdir(TRAINING_CATS_DIR)))
print(len(os.listdir(TRAINING_DOGS_DIR)))
print(len(os.listdir(TESTING_CATS_DIR)))
print(len(os.listdir(TESTING_DOGS_DIR)))

11249
11249
1250
1250


### defining the model  

In [7]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
]) 
model.compile(optimizer=RMSprop(lr=0.001), loss='binary_crossentropy', metrics=['accuracy'])

C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Argument(s) not recognized: {'lr': 0.001}

In [ ]:
import tensorflow as tf
from tensorflow.keras.optimizers import RMSprop

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(150, 150, 3)),
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer=RMSprop(learning_rate=0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

### train the model 

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TRAINING_DIR = os.path.join(split_root, "training")
VALIDATION_DIR = os.path.join(split_root, "testing")

train_datagen = ImageDataGenerator(rescale=1.0/255.)
train_generator = train_datagen.flow_from_directory(
    TRAINING_DIR,
    batch_size=100,
    class_mode='binary',
    target_size=(150, 150))

validation_datagen = ImageDataGenerator(rescale=1.0/255.)
validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    batch_size=100,
    class_mode='binary',
    target_size=(150, 150))

print(train_generator.class_indices)   # {'cats': 0, 'dogs': 1}


In [ ]:
# Note that this may take some time.
history = model.fit(
    train_generator,
    epochs=15,
    verbose=1,
    validation_data=validation_generator)

### exploring the result  

In [ ]:
%matplotlib inline
import matplotlib.image  as mpimg
import matplotlib.pyplot as plt
#-----------------------------------------------------------
# Retrieve a list of list results on training and test data
# sets for each training epoch
#-----------------------------------------------------------
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']
 
epochs=range(len(acc)) # Get number of epochs
 
#------------------------------------------------
# Plot training and validation accuracy per epoch
#------------------------------------------------
plt.plot(epochs, acc, 'r', "Training Accuracy")
plt.plot(epochs, val_acc, 'b', "Validation Accuracy")
plt.title('Training and validation accuracy')
plt.figure()
 
#------------------------------------------------
# Plot training and validation loss per epoch
#------------------------------------------------
plt.plot(epochs, loss, 'r', "Training Loss")
plt.plot(epochs, val_loss, 'b', "Validation Loss")
plt.figure()

### test 

In [ ]:
import numpy as np
import tkinter as tk
from tkinter import filedialog
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img, img_to_array

# Open a file picker (select one or more images)
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)   # keeps the dialog in front of your browser
file_paths = filedialog.askopenfilenames(
    title="Choose cat or dog images to test",
    filetypes=[("Images", "*.jpg *.jpeg *.png")]
)
root.destroy()

for path in file_paths:
    fn = os.path.basename(path)

    img = load_img(path, target_size=(150, 150))   # must match the model's input
    x = img_to_array(img)
    x = x / 255.0                                  # same normalization as training
    x = np.expand_dims(x, axis=0)                  # shape (1, 150, 150, 3)

    prediction = model.predict(x, verbose=0)[0][0]
    print(f"{fn}: score = {prediction:.3f}")

    if prediction > 0.5:
        print(fn + " is a dog\n")
    else:
        print(fn + " is a cat\n")

    plt.imshow(img)
    plt.axis('off')
    plt.show()

In [ ]:
import numpy as np
import tkinter as tk
from tkinter import filedialog
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img, img_to_array

# Open a file picker (select one or more images)
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)   # keeps the dialog in front of your browser
file_paths = filedialog.askopenfilenames(
    title="Choose cat or dog images to test",
    filetypes=[("Images", "*.jpg *.jpeg *.png")]
)
root.destroy()

for path in file_paths:
    fn = os.path.basename(path)

    img = load_img(path, target_size=(150, 150))   # must match the model's input
    x = img_to_array(img)
    x = x / 255.0                                  # same normalization as training
    x = np.expand_dims(x, axis=0)                  # shape (1, 150, 150, 3)

    prediction = model.predict(x, verbose=0)[0][0]
    print(f"{fn}: score = {prediction:.3f}")

    if prediction > 0.5:
        print(fn + " is a dog\n")
    else:
        print(fn + " is a cat\n")

    plt.imshow(img)
    plt.axis('off')
    plt.show()